In [1]:
# ============================================================================
# PART 1: INSTALLATION
# ============================================================================
print("="*70)
print("INSTALLING REQUIRED PACKAGES...")
print("="*70)

!pip uninstall -y peft trl transformers
!pip install -q transformers==4.45.0
!pip install -q peft==0.11.1
!pip install -q trl==0.9.6
!pip install -q bitsandbytes
!pip install -q accelerate
!pip install -q datasets

print("✅ Installation complete!\n")

INSTALLING REQUIRED PACKAGES...
Found existing installation: peft 0.16.0
Uninstalling peft-0.16.0:
  Successfully uninstalled peft-0.16.0
Found existing installation: transformers 4.53.3
Uninstalling transformers-4.53.3:
  Successfully uninstalled transformers-4.53.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 92.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/6

In [2]:
print("="*70)
print("IMPORTING LIBRARIES...")
print("="*70)

import os
import torch
import gc
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM
from trl import SFTTrainer

os.environ['WANDB_DISABLED'] = 'true'
torch.cuda.empty_cache()
gc.collect()

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\n")


IMPORTING LIBRARIES...


2025-12-08 11:24:19.891382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765193060.115214      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765193060.185674      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ GPU: Tesla T4
✅ GPU Memory: 15.83 GB



In [5]:
# ============================================================================
# PART 3: CONFIGURATION
# ============================================================================
print("="*70)
print("SETTING CONFIGURATION...")
print("="*70)

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
NEW_MODEL_NAME = "proposal-generator-llama3"
DATASET_NAME = "Proposal_Dataset.csv"  # replace with actual dataset ID

# Training hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
OUTPUT_DIR = "/kaggle/working/results"
EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048
SAVE_STEPS = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Model: {MODEL_NAME}")
print(f"✅ Output: {OUTPUT_DIR}")
print(f"✅ Epochs: {EPOCHS}\n")

SETTING CONFIGURATION...
✅ Model: unsloth/Llama-3.2-3B-Instruct
✅ Output: /kaggle/working/results
✅ Epochs: 3



In [7]:
# ============================================================================
# PART 4: LOAD DATASET FROM CSV
# ============================================================================
print("="*70)
print("LOADING DATASET FROM CSV...")
print("="*70)

import pandas as pd
from datasets import Dataset

csv_path = "/kaggle/input/proposal-data/Proposal_Dataset.csv"
df = pd.read_csv(csv_path)

print(f"✅ CSV loaded: {csv_path}")
print(f"Columns: {df.columns.tolist()}")
print(f"Number of samples: {len(df)}")
print(f"\nFirst row preview:\n{df.iloc[0]}")

# Convert to Hugging Face Dataset format for Trainer compatibility
dataset = Dataset.from_pandas(df)

print(f"✅ Dataset converted to HuggingFace Dataset")
print(f"Columns: {dataset.column_names}")
print(f"Number of samples: {len(dataset)}\n")

LOADING DATASET FROM CSV...
✅ CSV loaded: /kaggle/input/proposal-data/Proposal_Dataset.csv
Columns: ['instruction', 'prompt', 'output']
Number of samples: 90

First row preview:
instruction    Write a detailed project proposal for the foll...
prompt         System: You are an expert project manager. Fol...
output         Project Proposal: Adaptive learning platform u...
Name: 0, dtype: object
✅ Dataset converted to HuggingFace Dataset
Columns: ['instruction', 'prompt', 'output']
Number of samples: 90



In [ ]:
# ============================================================================
# PART 5: FORMAT DATASET FOR PROPOSAL GENERATION
# ============================================================================
print("="*70)
print("FORMATTING DATASET FOR PROPOSAL GENERATION...")
print("="*70)

def format_instruction(sample):
    """Format data into Llama-3 chat template for proposal generation"""
    instruction = sample['instruction']  # replace with actual column name if different
    output = sample['output']            # replace with actual column name if different

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert business and project proposal writer. Generate professional, compelling, and domain-specific proposals (education, healthcare, construction, business).<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

    return {"text": prompt}

formatted_dataset = dataset.map(
    format_instruction,
    remove_columns=dataset.column_names,
    desc="Formatting proposal samples"
)

print(f"✅ Dataset formatted successfully!")
print(f"✅ Total samples: {len(formatted_dataset)}")
print(f"\n📄 Formatted sample (first 400 chars):\n{formatted_dataset[0]['text'][:400]}...\n")

FORMATTING DATASET FOR PROPOSAL GENERATION...


Formatting proposal samples:   0%|          | 0/90 [00:00<?, ? examples/s]

✅ Dataset formatted successfully!
✅ Total samples: 90

📄 Formatted sample (first 400 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert business and project proposal writer. Generate professional, compelling, and domain-specific proposals (education, healthcare, construction, business).<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a detailed project proposal for the following initiative: Adaptive learning platform using AI.<|eot_id|><|start_...



In [ ]:
# ============================================================================
# PART 6: LOAD TOKENIZER
# ============================================================================
print("="*70)
print("LOADING TOKENIZER...")
print("="*70)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Tokenizer loaded")
print(f"✅ Vocab size: {len(tokenizer)}\n")

# ============================================================================
# PART 7: CONFIGURE 4-BIT QUANTIZATION
# ============================================================================
print("="*70)
print("CONFIGURING 4-BIT QUANTIZATION...")
print("="*70)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("✅ Quantization config ready\n")

# ============================================================================
# PART 8: LOAD BASE MODEL
# ============================================================================
print("="*70)
print("LOADING BASE MODEL...")
print("="*70)

from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

print(f"✅ Model loaded successfully")
print(f"✅ Total parameters: ~{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B\n")

# ============================================================================
# PART 9: CONFIGURE LORA
# ============================================================================
print("="*70)
print("CONFIGURING LoRA...")
print("="*70)

from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, peft_config)

print("✅ LoRA configured!")
model.print_trainable_parameters()

# ============================================================================
# PART 10: SETUP TRAINING ARGUMENTS
# ============================================================================
print("="*70)
print("SETTING UP TRAINING ARGUMENTS...")
print("="*70)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    learning_rate=LEARNING_RATE,
    bf16=False,
    tf32=False,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    load_best_model_at_end=False,
)

print(f"✅ Training arguments ready")
print(f"✅ Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}\n")

# ============================================================================
# PART 11: INITIALIZE TRAINER
# ============================================================================
print("="*70)
print("INITIALIZING TRAINER...")
print("="*70)

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
    dataset_text_field="text",
)

print("✅ Trainer ready\n")

# ============================================================================
# PART 12: TRAIN THE MODEL
# ============================================================================
print("="*70)
print("🚀 STARTING TRAINING...")
print("="*70)

trainer.train()

print("\n✅ Training complete!\n")

# ============================================================================
# PART 13: SAVE MODEL (PEFT ADAPTER)
# ============================================================================
print("="*70)
print("SAVING MODEL (PEFT ADAPTER)...")
print("="*70)

final_model_path = f"/kaggle/working/{NEW_MODEL_NAME}"
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Also save to results directory
trainer.model.save_pretrained(f"{OUTPUT_DIR}/{NEW_MODEL_NAME}")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/{NEW_MODEL_NAME}")

print(f"✅ PEFT adapter saved to: {final_model_path}\n")

# ============================================================================
# PART 13.5: MERGE BASE MODEL AND ADAPTER FOR FASTER INFERENCE
# ============================================================================
print("="*70)
print("🔄 MERGING BASE MODEL AND ADAPTER...")
print("="*70)
print("This creates a single merged model for faster loading during inference.\n")

# Path for merged model
# For Kaggle: save to working directory
# For local deployment: save to backend/ai/proposal_generator/model/merged
import os
kaggle_merged_path = f"/kaggle/working/{NEW_MODEL_NAME}-merged"
# Local merged path (adjust if running locally)
local_merged_path = "backend/ai/proposal_generator/model/merged"
merged_model_path = kaggle_merged_path  # Use Kaggle path by default

try:
    # Load the PEFT model
    print("[MERGE] Loading PEFT adapter model...")
    from peft import AutoPeftModelForCausalLM

    # Load base model + adapter
    peft_model = AutoPeftModelForCausalLM.from_pretrained(
        final_model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    print("[MERGE] Merging adapter weights into base model...")
    # Merge adapter with base model
    merged_model = peft_model.merge_and_unload()

        print("[MERGE] Saving merged model...")
    # Save merged model to Kaggle working directory
    merged_model.save_pretrained(merged_model_path)
    tokenizer.save_pretrained(merged_model_path)

    print(f"✅ Merged model saved to: {merged_model_path}")
    print(f"✅ This merged model can be loaded directly without base model!")
    print(f"✅ Loading will be faster since no adapter merging is needed.")

    # If running locally, also save to the merged folder
    if os.path.exists("backend"):
        print(f"\n[MERGE] Also saving to local merged folder...")
        os.makedirs(local_merged_path, exist_ok=True)
        merged_model.save_pretrained(local_merged_path)
        tokenizer.save_pretrained(local_merged_path)
        print(f"✅ Also saved to: {local_merged_path}\n")
    else:
        print(f"\n💡 To use in SoftScale, copy merged model to: backend/ai/proposal_generator/model/merged\n")

    # Clean up
    del peft_model
    del merged_model
    gc.collect()
    torch.cuda.empty_cache()

except Exception as e:
    print(f"⚠️  Warning: Could not merge model: {e}")
    print("   You can still use the PEFT adapter model.")
    print("   To merge later, use the merge_model.py script.\n")

# Free memory from training
import gc
import torch
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

# ============================================================================
# PART 14: LOAD MODEL FOR INFERENCE (USE MERGED MODEL IF AVAILABLE)
# ============================================================================
print("="*70)
print("LOADING MODEL FOR INFERENCE...")
print("="*70)

import os

merged_model_path = f"/kaggle/working/{NEW_MODEL_NAME}-merged"

# Try to load merged model first (faster)
if os.path.exists(merged_model_path) and os.path.exists(os.path.join(merged_model_path, "config.json")):
    print("[LOAD] Loading merged model (faster - no adapter needed)...")
    from transformers import AutoModelForCausalLM

    model = AutoModelForCausalLM.from_pretrained(
        merged_model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
    print("✅ Merged model loaded for inference (fast loading!)\n")
else:
    print("[LOAD] Loading PEFT adapter model (slower - requires base model)...")
    from peft import AutoPeftModelForCausalLM

    model = AutoPeftModelForCausalLM.from_pretrained(
        final_model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(final_model_path)
    print("✅ PEFT adapter model loaded for inference\n")

# ============================================================================
# PART 15: DEFINE PROPOSAL GENERATION FUNCTION
# ============================================================================
def generate_proposal(instruction, max_tokens=512, temperature=0.7):
    """Generate business or project proposal from instruction"""

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert business and project proposal writer.

Your task is to generate fully professional, compelling, and *highly varied* proposals in education, healthcare, construction, and business domains.

### STRICT REQUIREMENTS:
- Every proposal must be clear, well-structured, and professional.
- Use domain-specific vocabulary appropriately.
- Avoid repeating phrases; each proposal should look unique.
- Proposals must include objectives, methodology/approach, expected outcomes, and summary/recommendations.
- Maintain a polished, formal business tone.

Instruction:

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant<|end_header_id|>" in result:
        return result.split("assistant<|end_header_id|>")[-1].strip()
    return result

print("✅ Proposal generation function defined\n")


LOADING TOKENIZER...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

✅ Tokenizer loaded
✅ Vocab size: 128256

CONFIGURING 4-BIT QUANTIZATION...
✅ Quantization config ready

LOADING BASE MODEL...


...

✅ Trainer ready

🚀 STARTING TRAINING...


Step,Training Loss
5,2.036800
10,0.986600
15,0.456500



✅ Training complete!

SAVING MODEL...
✅ Model saved to: /kaggle/working/proposal-generator-llama3

LOADING MODEL FOR INFERENCE...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded for inference

✅ Proposal generation function defined



In [ ]:
# ============================================================================
# PART 16: TEST THE PROPOSAL GENERATOR
# ============================================================================
print("="*70)
print("🧪 TESTING THE PROPOSAL GENERATOR")
print("="*70 + "\n")

# ------------------------ TEST 1: Education Proposal ------------------------
print("-"*70)
print("TEST 1: Education Proposal")
print("-"*70 + "\n")

instruction_edu = """Create a detailed education project proposal for launching an online coding bootcamp targeting university students in Pakistan.
Include objectives, curriculum structure, teaching methodology, expected outcomes, and a summary with recommendations."""

proposal_edu = generate_proposal(instruction_edu)
print(proposal_edu)
print("\n")

# ------------------------ TEST 2: Healthcare Proposal ----------------------
print("-"*70)
print("TEST 2: Healthcare Proposal")
print("-"*70 + "\n")

instruction_health = """Generate a healthcare project proposal for setting up a mobile clinic to provide basic medical services in rural areas.
Include objectives, services offered, methodology, expected impact, and summary/recommendations."""

proposal_health = generate_proposal(instruction_health)
print(proposal_health)
print("\n")

# ------------------------ TEST 3: Construction Proposal --------------------
print("-"*70)
print("TEST 3: Construction Proposal")
print("-"*70 + "\n")

instruction_construction = """Write a construction project proposal for building a sustainable community center in an urban area.
Include objectives, design and construction approach, timeline, expected outcomes, and summary/recommendations."""

proposal_construction = generate_proposal(instruction_construction)
print(proposal_construction)
print("\n")

# ------------------------ TEST 4: Business Proposal ------------------------
print("-"*70)
print("TEST 4: Business Proposal")
print("-"*70 + "\n")

instruction_business = """Create a business proposal for launching a startup that provides AI-powered content generation services.
Include objectives, business model, implementation plan, expected outcomes, and summary/recommendations."""

proposal_business = generate_proposal(instruction_business)
print(proposal_business)
print("\n")

print("="*70)
print("✅ TESTING COMPLETE")
print("="*70 + "\n")


🧪 TESTING THE PROPOSAL GENERATOR

----------------------------------------------------------------------
TEST 1: Education Proposal
----------------------------------------------------------------------



Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


system

You are an expert business and project proposal writer.

Your task is to generate fully professional, compelling, and *highly varied* proposals in education, healthcare, construction, and business domains.

### STRICT REQUIREMENTS:
- Every proposal must be clear, well-structured, and professional.
- Use domain-specific vocabulary appropriately.
- Avoid repeating phrases; each proposal should look unique.
- Proposals must include objectives, methodology/approach, expected outcomes, and summary/recommendations.
- Maintain a polished, formal business tone.

Instruction:

Create a detailed education project proposal for launching an online coding bootcamp targeting university students in Pakistan. 
Include objectives, curriculum structure, teaching methodology, expected outcomes, and a summary with recommendations.assistant

Project Title: "Pakistan Coding Bootcamp Initiative"

Domain: Education
Objective:
- Launch an online coding bootcamp addressing the skill gap of Pakistani uni

In [ ]:
# ============================================================================
# PART 15 (UPDATED): DEFINE DYNAMIC GENERATION FUNCTION WITH ALL DOMAIN TEMPLATES
# ============================================================================
def generate_proposal(instruction, domain=None, max_tokens=700, temperature=0.7):
    """
    Generate a structured proposal dynamically based on domain.
    - domain: 'business', 'education', 'healthcare', 'construction'
    """

    # Default domain detection if not provided
    if domain is None:
        domain_lower = instruction.lower()
        if "business" in domain_lower or "startup" in domain_lower:
            domain = "business"
        elif "education" in domain_lower:
            domain = "education"
        elif "healthcare" in domain_lower or "clinic" in domain_lower:
            domain = "healthcare"
        elif "construction" in domain_lower or "building" in domain_lower:
            domain = "construction"
        else:
            domain = "general"

    # ------------------ DOMAIN SPECIFIC TEMPLATES ------------------

    if domain == "business":
        template = f"""
You are an expert professional business proposal writer.

### STRICT REQUIREMENTS:
1. Generate a **full, detailed, professional business proposal**.
2. Use **numbered sections and clear headings**.
3. Avoid repetitive words or phrases.
4. Include realistic details (metrics, timelines, resources) wherever relevant.
5. Maintain a polished, formal, professional tone.

### BUSINESS PROPOSAL TEMPLATE:
1. Project / Startup Name:
2. Executive Summary:
   - Overview of the project and key objectives (3–5 sentences)
3. Problem Statement / Opportunity:
   - Describe the market gap or opportunity
4. Objectives / Goals:
   - Objective 1:
   - Objective 2:
   - Objective 3:
5. Proposed Solution / Approach:
   - Detailed plan of action
6. Business Model / Revenue Plan:
   - How the project will generate revenue
7. Implementation Plan / Timeline:
   - Step-by-step activities and milestones
8. Expected Outcomes / Impact:
   - Tangible results and metrics
9. Summary & Recommendations / Call to Action:
   - Concluding remarks and next steps

### USER INSTRUCTION:
{instruction}
"""

    elif domain == "education":
        template = f"""
You are an expert educational proposal writer.

### STRICT REQUIREMENTS:
1. Generate a **full, detailed educational proposal**.
2. Use **bold headings** for all main sections.
3. Use numbered or bulleted lists for objectives, methodology, outcomes, and timelines.
4. Avoid repetition; write naturally and professionally.
5. Include realistic, actionable details for all sections.

### EDUCATIONAL PROPOSAL TEMPLATE:
**Title of Proposal:**
**Submitted By:**
**Date:**

**1. Executive Summary:**
(Overview of the project, purpose, goals, and expected outcomes in 4–6 sentences)

**2. Background / Rationale:**
(Problem statement, context, and justification for the project)

**3. Objectives:**
- Objective 1:
- Objective 2:
- Objective 3:

**4. Target Audience / Beneficiaries:**
-

**5. Proposed Methodology / Approach:**
-

**6. Timeline / Milestones:**
-

**7. Budget / Resources:**
-

**8. Expected Outcomes / Impact:**
-

**9. Evaluation & Monitoring:**
-

**10. Conclusion:**
-

**11. Appendices (Optional):**
-

### USER INSTRUCTION:
{instruction}
"""

    elif domain == "healthcare":
        template = f"""
You are an expert healthcare project proposal writer.

### STRICT REQUIREMENTS:
1. Generate a **detailed, professional healthcare proposal**.
2. Use **bold or numbered headings** for all main sections.
3. Include bullets/lists for objectives, methodology, services, outcomes.
4. Avoid repetition; write clearly and professionally.
5. Include realistic, actionable details, timelines, and metrics.

### HEALTHCARE PROPOSAL TEMPLATE:
**Project Name / Initiative:**
**Submitted By:**
**Date:**

**1. Executive Summary:**
(Brief overview of project purpose, goals, expected outcomes)

**2. Background / Rationale:**
(Explain the healthcare gap, context, and justification)

**3. Objectives:**
- Objective 1:
- Objective 2:
- Objective 3:

**4. Services Offered / Scope:**
-

**5. Proposed Methodology / Approach:**
-

**6. Timeline / Milestones:**
-

**7. Budget / Resources:**
-

**8. Expected Outcomes / Impact:**
-

**9. Evaluation & Monitoring:**
-

**10. Conclusion & Recommendations:**
-

**11. Appendices (Optional):**
-

### USER INSTRUCTION:
{instruction}
"""

    elif domain == "construction":
        template = f"""
You are an expert construction project proposal writer.

### STRICT REQUIREMENTS:
1. Generate a **full, detailed construction proposal**.
2. Use **bold or numbered headings** for all main sections.
3. Include bullets/lists for methodology, design, timeline, and budget.
4. Avoid repetition; maintain a formal professional tone.
5. Include realistic project details, resources, and measurable outcomes.

### CONSTRUCTION PROPOSAL TEMPLATE:
**Project Name / Initiative:**
**Submitted By:**
**Date:**

**1. Executive Summary:**
(Overview of project, purpose, goals, and expected outcomes)

**2. Background / Rationale:**
(Problem or need, context, justification)

**3. Objectives:**
- Objective 1:
- Objective 2:
- Objective 3:

**4. Design & Construction Approach:**
-

**5. Timeline / Milestones:**
-

**6. Budget / Resources:**
-

**7. Expected Outcomes / Impact:**
-

**8. Evaluation & Monitoring:**
-

**9. Conclusion & Recommendations:**
-

**10. Appendices (Optional):**
-

### USER INSTRUCTION:
{instruction}
"""

    else:
        template = f"""
You are an expert professional proposal writer.

### STRICT REQUIREMENTS:
1. Generate a detailed, professional proposal.
2. Use numbered sections or bold headings.
3. Avoid repetitive words and phrases.
4. Include realistic, actionable details wherever applicable.

### GENERAL PROPOSAL TEMPLATE:
1. Project Title / Name:
2. Executive Summary:
3. Background / Problem Statement:
4. Objectives / Goals:
5. Methodology / Approach:
6. Expected Outcomes / Impact:
7. Timeline / Milestones:
8. Budget / Resources:
9. Summary & Recommendations / Call to Action:

### USER INSTRUCTION:
{instruction}
"""

    # ------------------ PREPARE PROMPT ------------------
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{template}<|start_header_id|>assistant<|end_header_id|>\n"

    # Tokenize and send to model
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant<|end_header_id|>" in result:
        return result.split("assistant<|end_header_id|>")[-1].strip()
    return result


In [ ]:
# ============================================================================
# TEST 1: Education Proposal
# ============================================================================

test_1_instruction = """Create a detailed education project proposal for launching an online coding bootcamp targeting university students in Pakistan.
Include objectives, curriculum structure, teaching methodology, expected outcomes, timeline, and summary with recommendations."""

print("-"*70)
print("TEST 1: Education Proposal")
print("-"*70 + "\n")

result_1 = generate_proposal(test_1_instruction, domain="education")
print(result_1)


----------------------------------------------------------------------
TEST 1: Education Proposal
----------------------------------------------------------------------

system

You are an expert professional proposal writer in education.

### STRICT REQUIREMENTS:
1. Generate a formal, compelling proposal with clear headings.
2. Use numbered sections and bullets for objectives, methodology, and outcomes.
3. Maintain a professional business tone.

### PROPOSAL TEMPLATE:
1. Project Title / Name:
2. Domain: Education
3. Objectives:
   1. 
   2. 
4. Methodology / Approach:
   1. 
   2. 
5. Expected Outcomes:
   1. 
   2. 
6. Timeline / Milestones (if applicable):
   - 
7. Budget / Resources (if applicable):
   - 
8. Summary & Recommendations:
   - 

### USER INSTRUCTION:
Create a detailed education project proposal for launching an online coding bootcamp targeting university students in Pakistan. 
Include objectives, curriculum structure, teaching methodology, expected outcomes, timeline, 

In [ ]:
def generate_proposal_only(instruction, domain=None, max_tokens=700, temperature=0.7):
    """
    Generate only the final proposal text, skipping all system/template/instruction text.
    """
    proposal = generate_proposal(instruction, domain, max_tokens, temperature)

    # Attempt to find the start of the actual proposal by locating the first bold heading
    start_idx = proposal.find("**Title of Proposal:**")
    if start_idx != -1:
        proposal = proposal[start_idx:]

    # Remove any remaining template/instruction remnants
    for marker in ["### USER INSTRUCTION:", "(Overview of the project", "(Problem statement", "(Provide a concise title)", "(Your Name / Organization", "(Submission date)"]:
        proposal = proposal.replace(marker, "")

    # Clean extra newlines or whitespace
    proposal_lines = [line.strip() for line in proposal.splitlines() if line.strip()]
    return "\n".join(proposal_lines)


In [20]:
instruction = """
Create a detailed education project proposal for launching an online coding bootcamp
targeting university students in Pakistan. Include objectives, curriculum structure,
teaching methodology, expected outcomes, timeline, budget, evaluation, and a summary
with recommendations. The proposal must be professional, fully segmented, detailed,
and avoid repetitive words.
"""

education_proposal = generate_proposal_cleaned(instruction, domain="education")
print(education_proposal)


**Title of Proposal:**  
**Submitted By:**  
**Date:**  

**1. Executive Summary:**  
(Overview of the project, purpose, goals, and expected outcomes in 4–6 sentences)

**2. Background / Rationale:**  
(Problem statement, context, and justification for the project)

**3. Objectives:**  
- Objective 1:
- Objective 2:
- Objective 3:

**4. Target Audience / Beneficiaries:**  
- 

**5. Proposed Methodology / Approach:**  
- 

**6. Timeline / Milestones:**  
- 

**7. Budget / Resources:**  
- 

**8. Expected Outcomes / Impact:**  
- 

**9. Evaluation & Monitoring:**  
- 

**10. Conclusion:**  
- 

**11. Appendices (Optional):**  
- 

### USER INSTRUCTION:

Create a detailed education project proposal for launching an online coding bootcamp
targeting university students in Pakistan. Include objectives, curriculum structure,
teaching methodology, expected outcomes, timeline, budget, evaluation, and a summary
with recommendations. The proposal must be professional, fully segmented, detailed,
a

**Title of Proposal:** Launching "CodePak" - Online Coding Bootcamp for University Students in Pakistan
**Submitted By:** Muhammad Ali, Education Project Manager
**Date:** March 2023

**1. Executive Summary:**
This comprehensive project aims to establish "CodePak," an immersive online coding bootcamp designed specifically for university students in Pakistan. The initiative seeks to bridge the gap between theoretical knowledge and practical skills, fostering a community that drives innovation and digital literacy nationwide. Through a structured curriculum, mentorship programs, and real-world projects, participants will gain expertise in in-demand technologies. By the end of the program, alumni will be equipped to secure jobs in the tech industry or pursue advanced studies. Key outcomes include improved coding proficiency, employability, and network effects. Launched within six months, CodePak is poised to transform Pakistan's tech talent landscape.

**2. Background / Rationale:**
The Pakistani job market demands skilled professionals in technology and IT, yet the current education system often fails to equip graduates with practical skills. University students lack access to quality coding training, hindering their employability. Digital literacy gaps persist, affecting economic growth and national development. Addressing these challenges through targeted initiatives like CodePak is crucial for Pakistan's future prosperity.

**3. Objectives:**

1. Provide comprehensive coding training to 500 university students across 20 institutions within the first year.
2. Achieve a 75% job placement rate among participants within six months post-graduation.
3. Establish partnerships with top tech companies for internships and mentorship opportunities.
4. Develop a supportive community with regular hackathons, workshops, and online forums.

**4. Target Audience / Beneficiaries:**

* University students (CSE, ECE, CS)
* Educators seeking professional development opportunities
* Tech startups and industries looking for talent

**5. Proposed Methodology / Approach:**

1. Modular curriculum covering popular programming languages (Python, Java, JavaScript).
2. Interactive online sessions, live mentoring, and peer review.
3. Real-world projects and case studies for practical application.
4. Collaborations with industry experts for guest lectures and project guidance.
5. Mobile app-based platform for student engagement, tracking, and feedback.

**6. Timeline / Milestones:**

| Phase | Duration | Milestone |
| --- | --- | --- |
| Month 1-3 | Planning & Infrastructure Setup | Platform development, team formation, stakeholder engagement |
| Month 4-6 | Curriculum Development | Finalize modules, course content creation |
| Month 7-12 | Pilot Launch | Initial cohort launch, pilot testing, iteration |
| Month 13-24 | Full Scale Launch | Gradual expansion to more universities, marketing campaigns |

**7. Budget / Resources:**

1. Personnel (Development Team, Mentors, Support Staff) - PKR 15 million
2. Infrastructure (Server, Bandwidth, Software) - PKR 8 million
3. Marketing & Promotion - PKR 3 million
4. Partnerships & Collaborations - PKR 2 million
Total Budget: PKR 28 million

**8. Expected Outcomes / Impact:**

1. Improved coding skills among participants
2. Enhanced employability and job prospects
3. Network effects through community engagement and collaborations
4. Contribution to national digital literacy and skill development

**9. Evaluation & Monitoring:

In [21]:
import shutil

# Path to the folder you want to zip
folder_path = "/kaggle/working/proposal-generator-llama3"
zip_path = "/kaggle/working/proposal-generator-llama3.zip"

# Create a zip file
shutil.make_archive(base_name=zip_path.replace(".zip", ""), format='zip', root_dir=folder_path)

print(f"✅ Folder zipped successfully: {zip_path}")


✅ Folder zipped successfully: /kaggle/working/proposal-generator-llama3.zip
